In [1]:
# =========================================================
# 📦 IMPORTS & CONFIGURATION
# =========================================================
import requests
import pandas as pd

from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    when, col, make_date, lit, regexp_replace,
    upper, trim, broadcast
)

# ---------------------------
# Pipeline Configuration
# ---------------------------
START_YEAR = 2010
END_YEAR = 2025

CO2_BASE_URL = "https://www.eurocontrol.int/performance/data/download/csv"
COUNTRIES_API_URL = (
    "https://restcountries.com/v3.1/all"
    "?fields=name,cca2,cca3,region,subregion,latlng,population"
)

TABLE_NAME = "EU_co2_emissions"

StatementMeta(, 8ae3e524-e32c-42da-8b4b-c959af83aa2f, 3, Finished, Available, Finished, False)

In [2]:
# =========================================================
# ✈️ CO2 DATA INGESTION
# =========================================================
def load_co2_data(start_year: int, end_year: int) -> DataFrame:
    """Load multi-year CO2 data with schema drift handling."""

    combined_df = None

    for year in range(start_year, end_year + 1):
        file_url = f"{CO2_BASE_URL}/co2_emmissions_by_state_{year}.csv"

        try:
            pdf = pd.read_csv(file_url)

            # Handle schema drift
            if "FLIGHT_MONTH" not in pdf.columns:
                pdf["FLIGHT_MONTH"] = None

            pdf["YEAR_SOURCE"] = year

            spark_df = spark.createDataFrame(pdf)

            combined_df = (
                spark_df if combined_df is None
                else combined_df.unionByName(spark_df, allowMissingColumns=True)
            )

            print(f"✅ Loaded {year}")

        except Exception as error:
            print(f"❌ Failed {year}: {error}")

    if combined_df is None:
        raise ValueError("No data loaded.")

    return combined_df

co2_raw_df = load_co2_data(START_YEAR, END_YEAR)

StatementMeta(, 8ae3e524-e32c-42da-8b4b-c959af83aa2f, 4, Finished, Available, Finished, False)

✅ Loaded 2010
✅ Loaded 2011
✅ Loaded 2012
✅ Loaded 2013
✅ Loaded 2014
✅ Loaded 2015
✅ Loaded 2016
✅ Loaded 2017
✅ Loaded 2018
✅ Loaded 2019
✅ Loaded 2020
✅ Loaded 2021
✅ Loaded 2022
✅ Loaded 2023
✅ Loaded 2024
✅ Loaded 2025


In [3]:
# =========================================================
# 🔄 CO2 DATA TRANSFORMATION
# =========================================================
def transform_co2_data(df: DataFrame) -> DataFrame:
    """Clean and standardize CO2 dataset."""

    df_clean = (
        df
        .drop("FLIGHT_MONTH", "NOTE", "YEAR_SOURCE")
        .withColumnRenamed("TF", "FLIGHTS")

        # Clean & normalize once
        .withColumn(
            "STATE_NAME_CLEAN",
            upper(trim(regexp_replace(col("STATE_NAME"), " ", "")))
        )
        .withColumn(
            "STATE_CODE",
            regexp_replace(col("STATE_CODE"), " ", "")
        )
    )

    # 🔥 Apply mapping cleanly
    df_mapped = df_clean.withColumn(
        "STATE_NAME",
        #when(col("STATE_NAME_CLEAN") == "CANARYISLANDS", "CANARY")
        when(col("STATE_NAME_CLEAN") == "BOSNIAANDHERZEGOVINA", "BOSNIA AND HERZEGOVINA")
        .when(col("STATE_NAME_CLEAN") == "NORTHMACEDONIA", "NORTH MACEDONIA")
        .when(col("STATE_NAME_CLEAN") == "TÜRKIYE", "TURKEY")
        .when(col("STATE_NAME_CLEAN") == "MOLDOVA,REPUBLICOF", "MOLDOVA")
        .when(col("STATE_NAME_CLEAN") == "UNITEDKINGDOM", "UNITED KINGDOM")
        .otherwise(col("STATE_NAME_CLEAN"))
    )

    # 🔧 Final transformations
    df_final = (
        df_mapped
        .withColumn("CO2_QTY_TONNES", col("CO2_QTY_TONNES").cast("long"))
        .withColumn("DATE", make_date(col("YEAR"), col("MONTH"), lit(1)))
        .drop("STATE_NAME_CLEAN")  # cleanup temp column
        .select(
            "DATE",
            "STATE_NAME",
            "STATE_CODE",
            "CO2_QTY_TONNES",
            "FLIGHTS",
            "YEAR",
            "MONTH"
        )
    )

    return df_final


# Execute
co2_clean_df = transform_co2_data(co2_raw_df)

StatementMeta(, 8ae3e524-e32c-42da-8b4b-c959af83aa2f, 5, Finished, Available, Finished, False)

In [4]:
# =========================================================
# 🌍 GEOGRAPHY DATA INGESTION
# =========================================================
def fetch_country_data(api_url: str) -> DataFrame:
    """Fetch country metadata and return Spark DataFrame."""

    response = requests.get(
        api_url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )

    if response.status_code != 200:
        raise RuntimeError(f"API failed: {response.status_code}")

    data = response.json()

    records = [
        {
            "COUNTRY": c.get("name", {}).get("common"),
            "ISO2": c.get("cca2"),
            "ISO3": c.get("cca3"),
            "REGION": c.get("region"),
            "SUBREGION": c.get("subregion"),
            "LATITUDE": (c.get("latlng") or [None, None])[0],
            "LONGITUDE": (c.get("latlng") or [None, None])[1],
            "POPULATION": c.get("population")
        }
        for c in data if isinstance(c, dict)
    ]

    return (
        spark.createDataFrame(pd.DataFrame(records))
        .dropDuplicates()
        .filter(col("COUNTRY").isNotNull())
        .withColumn("COUNTRY", upper(trim(col("COUNTRY"))))
    )

geo_df = fetch_country_data(COUNTRIES_API_URL)

StatementMeta(, 8ae3e524-e32c-42da-8b4b-c959af83aa2f, 6, Finished, Available, Finished, False)

In [5]:
# =========================================================
# 🔗 DATA ENRICHMENT
# =========================================================
def enrich_co2_with_geo(co2_df: DataFrame, geo_df: DataFrame) -> DataFrame:
    """Join CO2 data with geography using broadcast optimization."""

    return (
        co2_df
        .join(broadcast(geo_df), co2_df["STATE_NAME"] == geo_df["COUNTRY"], "left")
        .select(
            co2_df["DATE"],
            co2_df["STATE_NAME"],
            co2_df["STATE_CODE"],
            co2_df["CO2_QTY_TONNES"],
            co2_df["FLIGHTS"],
            geo_df["ISO2"],
            geo_df["REGION"],
            geo_df["SUBREGION"],
            geo_df["LATITUDE"],
            geo_df["LONGITUDE"],
            geo_df["POPULATION"]
        )
    )

co2_enriched_df = enrich_co2_with_geo(co2_clean_df, geo_df)

StatementMeta(, 8ae3e524-e32c-42da-8b4b-c959af83aa2f, 7, Finished, Available, Finished, False)

In [6]:
# =========================================================
# 💾 LOAD TO LAKEHOUSE
# =========================================================
def write_to_lakehouse(df: DataFrame, table_name: str) -> None:
    """Write DataFrame to Fabric Lakehouse as Delta table."""

    (
        df
        .write
        .format("delta")
        .mode("overwrite")   # change to "append" in production
        .saveAsTable(table_name)
    )

    print(f"✅ Data written to table: {table_name}")

write_to_lakehouse(co2_enriched_df, TABLE_NAME)

display(co2_enriched_df)

StatementMeta(, 8ae3e524-e32c-42da-8b4b-c959af83aa2f, 8, Finished, Available, Finished, False)

✅ Data written to table: EU_co2_emissions


SynapseWidget(Synapse.DataFrame, 245b1f80-04bf-4aa0-9bbf-827052e5f7bc)

In [7]:
%%sql
SELECT DISTINCT STATE_NAME 
FROM eu_co2_emissions
WHERE REGION IS NULL

StatementMeta(, 8ae3e524-e32c-42da-8b4b-c959af83aa2f, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>